# Evaluate the fine-tuned adapter — Dataset & Column Description

Runs the SFT+DPO adapter on the **held-out test datasets** (from `splits.json`) for both tasks and scores predictions against the gold descriptions with:

- **ROUGE-1/2/L** — lexical overlap (CPU, no download)
- **BERTScore** — semantic similarity (downloads an embedding model)
- **Length / verbosity ratio** — generated tokens ÷ reference tokens, flagged above the proposal's brevity threshold

Reuses the **same** task builders and split as `finetune_descriptions.ipynb`, so test inputs match training inputs exactly.

**Where this runs:** generation needs the GPU; the metric math is CPU-light. Set `ADAPTER_PATH = None` to evaluate the untuned base model as a baseline.

## 0. Install (GPU host only)

In [ ]:
%pip install -q --upgrade "transformers>=4.51" "peft>=0.13" "bitsandbytes>=0.45" "datasets>=2.20" "accelerate>=0.34" "rouge-score>=0.1.2" "bert-score>=0.3.13" pandas

## 1. Configuration

In [ ]:
# @dryrun
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-8B"
ADAPTER_PATH = (
    "adapters/qwen3-8b-desc-dpo"  # set to None to evaluate the untuned base model
)
METADATA_PATH = Path("all_metadata.json")
SPLITS_PATH = Path("splits.json")
PRED_PATH = Path("eval_predictions.jsonl")
RESULTS_PATH = Path("eval_results.json")

# generation
MAX_NEW_TOKENS = {"dataset_description": 256, "column_description": 64}

# metric toggles
DO_ROUGE, DO_BERTSCORE, DO_LENGTH = True, True, True
BERTSCORE_MODEL = "roberta-large"  # swap for a smaller model if bandwidth is tight
VERBOSITY_THRESHOLD = 1.15  # flag gen/ref length ratio above this

# limit examples while smoke-testing (None = all test examples)
MAX_EVAL_PER_TASK = None
print("Eval config loaded. Adapter:", ADAPTER_PATH)

In [ ]:
# --- Colab setup: locate inputs + persist outputs (use the SAME PROJECT_DIR as the finetune notebook) ---
from pathlib import Path

try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = IN_COLAB  # set False to use Colab's local /content disk instead of Drive
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path(
        "/content/drive/MyDrive/MetadataLearn"
    )  # <-- same Drive folder used for training
else:
    PROJECT_DIR = Path(".")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = PROJECT_DIR / "all_metadata.json"
SPLITS_PATH = PROJECT_DIR / "splits.json"
PRED_PATH = PROJECT_DIR / "eval_predictions.jsonl"
RESULTS_PATH = PROJECT_DIR / "eval_results.json"
ADAPTER_PATH = str(
    PROJECT_DIR / "adapters" / "qwen3-8b-desc-dpo"
)  # or set to None for the base-model baseline

# eval needs both the metadata and the split written by the finetune notebook
if (not METADATA_PATH.exists() or not SPLITS_PATH.exists()) and IN_COLAB:
    print("Upload all_metadata.json and/or splits.json:")
    from google.colab import files

    up = files.upload()
    for nm in ("all_metadata.json", "splits.json"):
        if nm in up:
            (PROJECT_DIR / nm).write_bytes(up[nm])

assert (
    METADATA_PATH.exists() and SPLITS_PATH.exists()
), f"Need all_metadata.json + splits.json in {PROJECT_DIR}"
print("Inputs:", METADATA_PATH.name, "+", SPLITS_PATH.name, "| adapter:", ADAPTER_PATH)

## 2. Task definitions (identical to the finetune notebook)

In [ ]:
# @dryrun  — shared task definitions (identical in finetune + evaluate notebooks)
import json, random

SYSTEM_PROMPT = (
    "You are a precise technical writer for open-data catalogs. "
    "Write grounded, concise documentation using only the provided metadata. "
    "Do not speculate or invent statistics. Output only the requested text, "
    "with no preamble, headers, or markdown."
)

MIN_DATASET_DESC_CHARS = (
    40  # skip datasets whose gold description is too thin to learn from
)
MIN_COLUMN_DESC_CHARS = 10  # skip columns with no real description
MAX_SAMPLES_DATASET = (
    4  # sample values shown per column in the dataset-description prompt
)
MAX_SAMPLES_COLUMN = 8  # sample values shown in the column-description prompt
MAX_COLS_IN_PROMPT = 40  # cap columns listed in the dataset-description prompt


def _fmt_samples(samples, n):
    vals = [str(s) for s in (samples or [])[:n] if str(s).strip()]
    return ", ".join(vals) if vals else "(no samples)"


def _column_block(columns):
    lines = []
    for c in columns[:MAX_COLS_IN_PROMPT]:
        lines.append(
            f"- {c.get('name','')} ({c.get('type','')}): e.g. "
            f"{_fmt_samples(c.get('samples'), MAX_SAMPLES_DATASET)}"
        )
    if len(columns) > MAX_COLS_IN_PROMPT:
        lines.append(f"- ... (+{len(columns) - MAX_COLS_IN_PROMPT} more columns)")
    return "\n".join(lines)


def build_dataset_desc_example(rec):
    """One dataset-description training example, or None if the dataset is unusable."""
    desc = (rec.get("description") or "").strip()
    cols = rec.get("columns") or []
    if len(desc) < MIN_DATASET_DESC_CHARS or len(cols) < 2:
        return None
    user = (
        "Task: Write a concise description (2-4 sentences) for the following dataset, "
        "suitable for an open-data catalog. Use only the provided schema and sample values.\n\n"
        f"Dataset name: {rec.get('name','')}\n"
        f"Columns:\n{_column_block(cols)}\n\n"
        "Description:"
    )
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]
    return {
        "uid": rec.get("uid"),
        "task": "dataset_description",
        "column": None,
        "prompt_messages": prompt_messages,
        "messages": prompt_messages + [{"role": "assistant", "content": desc}],
        "target": desc,
    }


def build_column_desc_examples(rec):
    """All column-description training examples for one dataset."""
    out = []
    name = rec.get("name", "")
    ddesc = (rec.get("description") or "").strip()
    ddesc_short = (
        (ddesc[:300] + "...") if len(ddesc) > 300 else (ddesc or "(none provided)")
    )
    for c in rec.get("columns") or []:
        cdesc = (c.get("description") or "").strip()
        if len(cdesc) < MIN_COLUMN_DESC_CHARS:
            continue
        user = (
            "Task: Write a one-sentence description of the column named below, for the given "
            "dataset. Use only the provided context and sample values.\n\n"
            f"Dataset: {name}\n"
            f"Dataset description: {ddesc_short}\n"
            f"Column name: {c.get('name','')}\n"
            f"Column type: {c.get('type','')}\n"
            f"Sample values: {_fmt_samples(c.get('samples'), MAX_SAMPLES_COLUMN)}\n\n"
            "Column description:"
        )
        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ]
        out.append(
            {
                "uid": rec.get("uid"),
                "task": "column_description",
                "column": c.get("name"),
                "prompt_messages": prompt_messages,
                "messages": prompt_messages + [{"role": "assistant", "content": cdesc}],
                "target": cdesc,
            }
        )
    return out


def build_examples_for_uids(records, uids):
    """Return (dataset_desc_examples, column_desc_examples) for the given uids."""
    ds_ex, col_ex = [], []
    for uid in uids:
        rec = records[uid]
        d = build_dataset_desc_example(rec)
        if d:
            ds_ex.append(d)
        col_ex.extend(build_column_desc_examples(rec))
    return ds_ex, col_ex


def split_uids(uids, seed=42, val_frac=0.10, test_frac=0.10):
    """Deterministic held-out-by-dataset split so columns never leak across splits."""
    uids = sorted(uids)
    rng = random.Random(seed)
    rng.shuffle(uids)
    n = len(uids)
    n_test = max(1, round(n * test_frac))
    n_val = max(1, round(n * val_frac))
    return {
        "test": sorted(uids[:n_test]),
        "val": sorted(uids[n_test : n_test + n_val]),
        "train": sorted(uids[n_test + n_val :]),
    }

## 3. Load test split + build eval examples

In [ ]:
# @dryrun
records = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
splits = json.loads(SPLITS_PATH.read_text(encoding="utf-8"))
print({k: len(v) for k, v in splits.items()})

test_ds_ex, test_col_ex = build_examples_for_uids(records, splits["test"])
if MAX_EVAL_PER_TASK:
    test_ds_ex = test_ds_ex[:MAX_EVAL_PER_TASK]
    test_col_ex = test_col_ex[:MAX_EVAL_PER_TASK]
eval_examples = test_ds_ex + test_col_ex
print(
    f"test datasets: {len(splits['test'])} | "
    f"dataset_description: {len(test_ds_ex)} | column_description: {len(test_col_ex)}"
)

## 4. Load base + adapter *(GPU)*

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True,  # allow CPU offload for modules that don't fit in GPU
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
if ADAPTER_PATH:
    from peft import PeftModel

    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    print("Loaded adapter:", ADAPTER_PATH)
else:
    print("Evaluating untuned base model (baseline)")
model.eval()

## 5. Generate predictions *(GPU)*

In [ ]:
import json, re, time

_THINK_RE = re.compile(
    r"^\s*<think>.*?</think>\s*", re.DOTALL
)  # defensive: drop any leading think block


def generate(prompt_messages, max_new_tokens):
    # enable_thinking=False matches the training-time render (empty think block in the prompt)
    try:
        text = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            prompt_messages, tokenize=False, add_generation_prompt=True
        )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    ).strip()
    return _THINK_RE.sub("", decoded).strip()


predictions = []
start_time = time.time()
for i, ex in enumerate(eval_examples, 1):
    pred = generate(ex["prompt_messages"], MAX_NEW_TOKENS[ex["task"]])
    predictions.append(
        {
            "uid": ex["uid"],
            "task": ex["task"],
            "column": ex["column"],
            "prediction": pred,
            "reference": ex["target"],
        }
    )
    if i % 10 == 0 or i == len(eval_examples):
        elapsed = time.time() - start_time
        rate = i / elapsed
        remaining = (len(eval_examples) - i) / rate if rate > 0 else 0
        print(
            f"  [{i:3d}/{len(eval_examples)}] ({rate:.1f} ex/s, ~{remaining:.0f}s remaining)"
        )

elapsed_total = time.time() - start_time
print(f"\nGenerated {len(predictions)} predictions in {elapsed_total:.1f}s\n")

with open(PRED_PATH, "w", encoding="utf-8") as f:
    for p in predictions:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print("wrote", PRED_PATH)

## 6. Metrics

Computed overall and broken down by task.

In [ ]:
from collections import defaultdict

by_task = defaultdict(list)
for p in predictions:
    by_task[p["task"]].append(p)
groups = {"overall": predictions, **by_task}

results = {}

# ---- ROUGE ----
if DO_ROUGE:
    from rouge_score import rouge_scorer

    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    for name, rows in groups.items():
        agg = defaultdict(float)
        for p in rows:
            s = scorer.score(p["reference"], p["prediction"])
            for k, v in s.items():
                agg[k] += v.fmeasure
        n = max(1, len(rows))
        results.setdefault(name, {}).update(
            {k: round(v / n, 4) for k, v in agg.items()}
        )

# ---- BERTScore ----
if DO_BERTSCORE:
    import bert_score

    for name, rows in groups.items():
        if name == "overall":
            continue  # scored per task; overall is the example-weighted mean below
        preds = [p["prediction"] for p in rows]
        refs = [p["reference"] for p in rows]
        _, _, f1 = bert_score.score(
            preds,
            refs,
            lang="en",
            model_type=BERTSCORE_MODEL,
            rescale_with_baseline=True,
        )
        results.setdefault(name, {})["bertscore_f1"] = round(float(f1.mean()), 4)
    # example-weighted overall
    tot = sum(results[t].get("bertscore_f1", 0) * len(by_task[t]) for t in by_task)
    results.setdefault("overall", {})["bertscore_f1"] = round(
        tot / max(1, len(predictions)), 4
    )

# ---- Length / verbosity ratio ----
if DO_LENGTH:

    def n_tok(s):
        return len(tokenizer.encode(s, add_special_tokens=False))

    for name, rows in groups.items():
        ratios = []
        for p in rows:
            r = n_tok(p["reference"])
            ratios.append(n_tok(p["prediction"]) / r if r else 0.0)
        n = max(1, len(ratios))
        results.setdefault(name, {}).update(
            {
                "len_ratio_mean": round(sum(ratios) / n, 3),
                "pct_over_threshold": round(
                    100 * sum(x > VERBOSITY_THRESHOLD for x in ratios) / n, 1
                ),
            }
        )

print(json.dumps(results, indent=2))

## 7. Results table + save

In [ ]:
import pandas as pd

df = pd.DataFrame(results).T
df.index.name = "group"
RESULTS_PATH.write_text(
    json.dumps(
        {
            "model_id": MODEL_ID,
            "adapter": ADAPTER_PATH,
            "n_examples": len(predictions),
            "verbosity_threshold": VERBOSITY_THRESHOLD,
            "metrics": results,
        },
        indent=2,
    ),
    encoding="utf-8",
)
print("wrote", RESULTS_PATH, "\n")
df

## 8. Inspect predictions vs gold vs base (non-fine-tuned) model

Shows, side by side, the fine-tuned adapter's output, the **untuned base model's** output for the *identical* prompt, and the gold reference. When an adapter is loaded we temporarily disable it (`model.disable_adapter()`) to get the base-model output — no second model load needed. If `ADAPTER_PATH` is `None`, the main run already *is* the base model, so the base column mirrors it.

In [ ]:
# Compare the fine-tuned adapter against the gold AND the untuned base model.
# `predictions` and `eval_examples` are built in the same order, so we zip them
# to recover each prediction's prompt_messages for the base-model re-generation.
has_adapter = bool(ADAPTER_PATH) and hasattr(model, "disable_adapter")
if not has_adapter:
    print("No adapter loaded — base column mirrors the main predictions.\n")

# How many examples to show per task (more column examples since they're short).
N_SHOW = {"dataset_description": 3, "column_description": 10}

for task in ("dataset_description", "column_description"):
    pairs = [(ex, p) for ex, p in zip(eval_examples, predictions) if p["task"] == task][
        : N_SHOW[task]
    ]
    print(f"\n########## {task} ({len(pairs)} shown) ##########")
    for ex, p in pairs:
        tag = f"{p['uid']}" + (f"/{p['column']}" if p["column"] else "")
        # Base (non-fine-tuned) output for the identical prompt.
        if has_adapter:
            with model.disable_adapter():
                base_pred = generate(ex["prompt_messages"], MAX_NEW_TOKENS[task])
        else:
            base_pred = p["prediction"]
        print(f"\n[{tag}]")
        print("BASE      :", base_pred[:300])
        print("FINE-TUNED:", p["prediction"][:300])
        print("GOLD      :", p["reference"][:300])